In [1]:
import tensorflow as tf
import os
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import json

"""
PHYSICS-CONSTRAINED OPEN-SET LIVESTOCK IDENTIFICATION

Final Objective Function (6 Components):
L = L_arc + λ1·L_center + λ2·L_unknown + λ3·L_sym + λ4·L_energy + λ5·L_lap

Components:
- L_arc: ArcFace angular margin loss
- L_center: Center loss for intra-class compactness  
- L_unknown: Unknown rejection loss
- L_sym: Symmetry constraint (flip invariance)
- L_energy: Energy margin loss (per-sample, pre-L2)
- L_lap: Smoothness constraint (perturbation robustness)
"""

# Set seeds
np.random.seed(42)
tf.random.set_seed(42)

# GPU setup
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except:
        pass

2026-02-22 18:35:43.649665: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-22 18:35:44.127800: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-22 18:35:44.521598: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-22 18:35:48.726743: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; 

In [2]:
dataset_root = "/home/student/Desktop/Kumar/dataset"

In [3]:
# CONFIGURATION


class CFG:
    IMG_SIZE = 224
    EMBED_DIM = 512
    MARGIN = 0.5
    SCALE = 64
    BATCH_SIZE = 32
    
    # Loss weights
    CENTER_LOSS_WEIGHT = 0.0001
    UNKNOWN_LOSS_WEIGHT = 0.1
    
    # Physics loss weights (final values after warmup)
    PHYSICS_SYM_WEIGHT = 0.01      # L_sym - Symmetry
    PHYSICS_ENERGY_WEIGHT = 0.01   # L_energy - Energy Margin
    PHYSICS_LAP_WEIGHT = 0.01      # L_lap - Smoothness
    
    # Physics warmup
    PHYSICS_WARMUP_EPOCHS = 10
    
    # Energy configuration
    ENERGY_TAU = 0.1               # Temperature for LogSumExp
    ENERGY_MARGIN = 0.2            # Per-sample margin
    
    STEPS_PER_EPOCH = 200
    AUTOTUNE = tf.data.AUTOTUNE
    TRAIN_SPLIT = 0.85  # 85% train, 15% val per cow
    
    # Curriculum schedule
    PHASE_1_EPOCHS = 5   # Warmup: 100% known
    PHASE_2_EPOCHS = 15  # Introduction: 80% known, 20% unknown
    PHASE_3_EPOCHS = 10  # Hardening: 60% known, 40% unknown
    
    TOTAL_EPOCHS = PHASE_1_EPOCHS + PHASE_2_EPOCHS + PHASE_3_EPOCHS

print("PHYSICS-CONSTRAINED TRAINING")

PHYSICS-CONSTRAINED TRAINING


In [4]:
# DATA LOADING


def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

def augment_image(img):
    """Standard augmentation for known samples"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.25)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    return img

def augment_unknown(img):
    """Heavy augmentation to create synthetic unknowns"""
    img = tf.image.random_brightness(img, 0.5)
    img = tf.image.random_contrast(img, 0.5, 1.5)
    img = tf.image.random_saturation(img, 0.5, 1.5)
    img = tf.image.random_hue(img, 0.2)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=0.1)
    img = img + noise
    img = tf.clip_by_value(img, -2.0, 2.0)
    return img

class CombinedDatasetBuilder:
    """Combines all cows from face_train, face_test, face_val"""
    def __init__(self, face_folders, split='train', train_split=0.85, seed=42):
        self.cow_data = {}
        self.split = split
        
        np.random.seed(seed)
        
        # Combine all folders
        for face_folder in face_folders:
            if not os.path.exists(face_folder):
                print(f"  Warning: {face_folder} not found, skipping...")
                continue
            
            for cls in sorted(os.listdir(face_folder)):
                cls_path = os.path.join(face_folder, cls)
                if not os.path.isdir(cls_path):
                    continue
                
                imgs = [os.path.join(cls_path, f) for f in os.listdir(cls_path)
                       if f.lower().endswith((".jpg", ".jpeg", ".png"))]
                
                if len(imgs) > 0:
                    if cls in self.cow_data:
                        self.cow_data[cls].extend(imgs)
                    else:
                        self.cow_data[cls] = imgs
        
        # Split each cow's images into train/val
        self.split_data = {}
        for cow_id, img_list in self.cow_data.items():
            img_list_shuffled = sorted(img_list)  # Sort for reproducibility
            indices = np.arange(len(img_list_shuffled))
            np.random.shuffle(indices)
            img_list_shuffled = [img_list_shuffled[i] for i in indices]
            
            n_train = int(len(img_list_shuffled) * train_split)
            
            if split == 'train':
                self.split_data[cow_id] = img_list_shuffled[:n_train]
            else:  # 'val'
                self.split_data[cow_id] = img_list_shuffled[n_train:]
        
        self.classes = sorted(list(self.split_data.keys()))
        self.cls_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.split_data.values())
    
    def get_tfdata_unbatched(self, shuffle=True, augment=False):
        face_paths = []
        labels = []
        
        for cow_id, face_list in self.split_data.items():
            for face_path in face_list:
                face_paths.append(face_path)
                labels.append(self.cls_to_idx[cow_id])
        
        ds = tf.data.Dataset.from_tensor_slices({
            'face_path': face_paths,
            'label': labels
        })
        
        def map_fn(sample):
            face_img = load_image(sample['face_path'])
            
            if augment:
                face_img = augment_image(face_img)
            
            return {
                'face_input': face_img,
                'label_input': sample['label']
            }, sample['label']
        
        ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
        
        if shuffle:
            ds = ds.shuffle(2000, seed=42, reshuffle_each_iteration=True)
        
        ds = ds.repeat()
        return ds

# Load data

print("LOADING DATA")


all_face_folders = [
    os.path.join(dataset_root, "face_train"),
    os.path.join(dataset_root, "face_test"),
    os.path.join(dataset_root, "face_val")
]

print(f"\nCombining data from:")
for folder in all_face_folders:
    print(f"  - {folder}")

train_data = CombinedDatasetBuilder(all_face_folders, split='train', train_split=CFG.TRAIN_SPLIT)
val_data = CombinedDatasetBuilder(all_face_folders, split='val', train_split=CFG.TRAIN_SPLIT)

print(f"\n✓ Loaded {train_data.num_classes} cattle (ALL cows combined)")
print(f"  Training:   {train_data.total_images} images ({CFG.TRAIN_SPLIT*100:.0f}%)")
print(f"  Validation: {val_data.total_images} images ({(1-CFG.TRAIN_SPLIT)*100:.0f}%)")
print(f"  All cattle appear in both train and val splits")

LOADING DATA

Combining data from:
  - /home/student/Desktop/Kumar/dataset/face_train
  - /home/student/Desktop/Kumar/dataset/face_test
  - /home/student/Desktop/Kumar/dataset/face_val

✓ Loaded 468 cattle (ALL cows combined)
  Training:   3774 images (85%)
  Validation: 952 images (15%)
  All cattle appear in both train and val splits


In [5]:
# MODEL ARCHITECTURE


def build_model(num_classes, embedding_size=512, margin=0.5, scale=64):
    """Build model with named embedding layer"""
    face_input = keras.Input(shape=(224, 224, 3), name="face_input")
    label_input = keras.Input(shape=(), dtype=tf.int32, name="label_input")
    
    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        input_shape=(224, 224, 3),
        weights="imagenet"
    )
    
    for layer in backbone.layers[:-30]:
        layer.trainable = False
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    
    x = backbone(face_input, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    
    # IMPORTANT: Unnormalized embeddings (for energy computation)
    x_unnorm = layers.Dense(embedding_size, kernel_regularizer=keras.regularizers.l2(1e-4),
                            name="embedding_dense")(x)
    
    # Normalized embeddings (for ArcFace)
    embeddings = layers.Lambda(
        lambda x: tf.math.l2_normalize(x, axis=1),
        name="l2_normalize"
    )(x_unnorm)
    
    class SafeArcFaceLayer(layers.Layer):
        def __init__(self, n_classes, margin=0.5, scale=64, **kwargs):
            super().__init__(**kwargs)
            self.n_classes = n_classes
            self.margin = margin
            self.scale = scale
            self.cos_m = tf.cos(margin)
            self.sin_m = tf.sin(margin)
        
        def build(self, input_shape):
            self.W = self.add_weight(
                name="arcface_weights",
                shape=(input_shape[0][-1], self.n_classes),
                initializer=keras.initializers.HeNormal(),
                trainable=True,
                regularizer=keras.regularizers.l2(1e-4)
            )
            super().build(input_shape)
        
        def call(self, inputs):
            embeddings, labels = inputs
            
            W_norm = tf.nn.l2_normalize(self.W, axis=0)
            embeddings_norm = tf.nn.l2_normalize(embeddings, axis=1)
            
            cos_theta = tf.matmul(embeddings_norm, W_norm)
            cos_theta = tf.clip_by_value(cos_theta, -1.0 + 1e-7, 1.0 - 1e-7)
            
            sin_theta = tf.sqrt(tf.maximum(1.0 - tf.square(cos_theta), 1e-7))
            target_logits = cos_theta * self.cos_m - sin_theta * self.sin_m
            
            labels = tf.cast(labels, tf.int32)
            known_mask = tf.greater_equal(labels, 0)
            labels_safe = tf.where(known_mask, labels, tf.zeros_like(labels))
            
            onehot_raw = tf.one_hot(labels_safe, depth=self.n_classes, dtype=tf.float32)
            onehot = onehot_raw * tf.expand_dims(tf.cast(known_mask, tf.float32), axis=1)
            
            output = cos_theta * (1.0 - onehot) + target_logits * onehot
            output = output * self.scale
            
            return output
        
        def get_config(self):
            return {"n_classes": self.n_classes, "margin": self.margin, "scale": self.scale}
    
    logits = SafeArcFaceLayer(num_classes, margin=margin, scale=scale,
                              name="arcface")([embeddings, label_input])
    
    # Model outputs BOTH unnormalized and normalized embeddings
    model = keras.Model(
        inputs=[face_input, label_input],
        outputs=[logits, embeddings, x_unnorm],  # Added x_unnorm for energy
        name="physics_constrained_model_improved"
    )
    
    return model

print("BUILDING MODEL")

model = build_model(
    train_data.num_classes,
    embedding_size=CFG.EMBED_DIM,
    margin=CFG.MARGIN,
    scale=CFG.SCALE
)

print(f"\n Model built for {train_data.num_classes} cattle")
print(f"  Architecture: EfficientNetB0 → Dense(512) → L2-Normalize → ArcFace")

BUILDING MODEL


2026-02-22 18:37:54.869899: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-22 18:37:57.033760: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14455 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:d8:00.0, compute capability: 7.5



 Model built for 468 cattle
  Architecture: EfficientNetB0 → Dense(512) → L2-Normalize → ArcFace


In [6]:
# Embedding-only model
embedding_model = keras.Model(
    inputs=model.input[0],
    outputs=[model.output[1], model.output[2]],  # Both normalized and unnormalized
    name="embedding_extractor"
)

print(f" Embedding extraction model created")

 Embedding extraction model created


In [8]:
# LOSSES

class CenterLossLayer(layers.Layer):
    """Center loss with training-aware updates"""
    def __init__(self, num_classes, embedding_dim, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim
        self.alpha = tf.constant(alpha, dtype=tf.float32)
    
    def build(self, input_shape):
        self.centers = self.add_weight(
            name="class_centers",
            shape=(self.num_classes, self.embedding_dim),
            initializer="zeros",
            trainable=False
        )
        super().build(input_shape)
    
    def call(self, embeddings, labels, training=True):
        labels = tf.cast(labels, tf.int32)
        known_mask = labels >= 0
        known_embeddings = tf.boolean_mask(embeddings, known_mask)
        known_labels = tf.boolean_mask(labels, known_mask)
        
        if tf.shape(known_embeddings)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)
        
        centers_batch = tf.gather(self.centers, known_labels)
        diff = known_embeddings - centers_batch
        loss = tf.reduce_mean(tf.reduce_sum(tf.square(diff), axis=1))
        
        if not training:
            return loss
        
        # Update centers (only during training)
        unique_labels, idx = tf.unique(known_labels)
        num_unique = tf.shape(unique_labels)[0]
        summed = tf.math.unsorted_segment_sum(known_embeddings, idx, num_unique)
        counts = tf.math.unsorted_segment_sum(tf.ones_like(idx, dtype=tf.float32), idx, num_unique)
        counts = tf.reshape(counts, (-1, 1))
        new_centers = summed / (counts + 1e-8)
        
        current_centers = tf.gather(self.centers, unique_labels)
        updated = self.alpha * current_centers + (1.0 - self.alpha) * new_centers
        
        scatter_idx = tf.expand_dims(unique_labels, axis=1)
        self.centers.assign(tf.tensor_scatter_nd_update(self.centers, scatter_idx, updated))
        
        return loss

class UnknownRejectionLoss(layers.Layer):
    """Unknown rejection loss with frozen class weights"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def call(self, embeddings, labels, class_weights):
        labels = tf.cast(labels, tf.int32)
        unknown_mask = tf.less(labels, 0)
        unknown_embeddings = tf.boolean_mask(embeddings, unknown_mask)
        
        if tf.shape(unknown_embeddings)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)
        
        W_frozen = tf.stop_gradient(class_weights)
        W_norm = tf.nn.l2_normalize(W_frozen, axis=0)
        
        similarities = tf.matmul(unknown_embeddings, W_norm)
        max_similarities = tf.reduce_max(similarities, axis=1)
        loss = tf.reduce_mean(max_similarities)
        
        return loss

center_loss_layer = CenterLossLayer(train_data.num_classes, CFG.EMBED_DIM, alpha=0.5)
unknown_rejection_layer = UnknownRejectionLoss()

print(f"\n Loss layers initialized")


 Loss layers initialized


In [9]:
# IMPROVED PHYSICS LOSSES


def get_physics_weights_tf(epoch, warmup_epochs=10):
    """TF-native weight scheduler for all physics losses"""
    epoch = tf.cast(epoch, tf.float32)
    warmup_epochs = tf.cast(warmup_epochs, tf.float32)
    scale = tf.minimum(epoch / warmup_epochs, 1.0)
    
    return (
        CFG.PHYSICS_SYM_WEIGHT * scale,      # Symmetry
        CFG.PHYSICS_ENERGY_WEIGHT * scale,   # Energy margin
        CFG.PHYSICS_LAP_WEIGHT * scale       # Smoothness
    )

def compute_smooth_energy_improved(embeddings_unnorm, class_weights, tau=0.1):
    """
    IMPROVED: Compute energy on PRE-L2 embeddings
    
    This allows the energy to modulate scale, not just angles.
    Gradients flow through unnormalized space for stronger separation.
    
    Args:
        embeddings_unnorm: Unnormalized embeddings (before L2 norm)
        class_weights: ArcFace class prototypes
        tau: Temperature parameter
    
    Returns:
        energy: Scalar energy for each embedding
    """
    # Freeze class weights
    W_frozen = tf.stop_gradient(class_weights)
    W_norm = tf.nn.l2_normalize(W_frozen, axis=0)
    
    # Normalize embeddings INSIDE energy function
    z_norm = tf.nn.l2_normalize(embeddings_unnorm, axis=1)
    
    # Cosine similarities
    cos_sim = tf.matmul(z_norm, W_norm)
    cos_sim = cos_sim - tf.reduce_max(cos_sim, axis=1, keepdims=True)
    
    # LogSumExp
    exp_term = tf.exp(cos_sim / tau)
    log_sum = tf.math.log(tf.reduce_sum(exp_term, axis=1) + 1e-8)
    energy = -tau * log_sum
    
    return energy

def energy_margin_loss_improved(known_embeddings_unnorm, unknown_embeddings_unnorm,
                                class_weights, margin=0.2, tau=0.1):
    """
    IMPROVED: Per-sample energy margin loss
    
    Instead of comparing batch means, enforce:
    E(unknown_i) > E(known_j) + margin for ALL pairs
    
    This is much stronger and prevents individual failures.
    
    Args:
        known_embeddings_unnorm: Unnormalized embeddings of known samples
        unknown_embeddings_unnorm: Unnormalized embeddings of unknown samples
        class_weights: ArcFace prototypes
        margin: Energy margin
        tau: Temperature
    
    Returns:
        loss: Per-sample margin loss
        E_known_mean: Mean energy of known (monitoring)
        E_unknown_mean: Mean energy of unknown (monitoring)
    """
    # Compute energies on PRE-L2 embeddings
    E_known = compute_smooth_energy_improved(known_embeddings_unnorm, class_weights, tau)
    E_unknown = compute_smooth_energy_improved(unknown_embeddings_unnorm, class_weights, tau)
    
    # Per-sample pairwise margin
    # Shape: (n_unknown, 1) - (1, n_known) = (n_unknown, n_known)
    E_unknown_expanded = tf.expand_dims(E_unknown, 1)
    E_known_expanded = tf.expand_dims(E_known, 0)
    
    pairwise_diff = E_unknown_expanded - E_known_expanded
    
    # Margin loss: want every unknown > every known + margin
    loss = tf.reduce_mean(tf.maximum(0.0, margin - pairwise_diff))
    
    # For monitoring
    E_known_mean = tf.reduce_mean(E_known)
    E_unknown_mean = tf.reduce_mean(E_unknown)
    
    return loss, E_known_mean, E_unknown_mean

print(f" Physics loss functions defined")

 Physics loss functions defined


In [10]:
# TRAINER


class ImprovedPhysicsTrainer(keras.Model):
    """Improved trainer with expert feedback fixes"""
    
    def __init__(self, base_model, embedding_model, center_loss_layer,
                 unknown_rejection_layer, center_loss_weight, unknown_loss_weight,
                 physics_warmup_epochs, **kwargs):
        super().__init__(**kwargs)
        
        self.base_model = base_model
        self.embedding_model = embedding_model
        self.center_loss_layer = center_loss_layer
        self.unknown_rejection_layer = unknown_rejection_layer
        
        self.center_loss_weight = center_loss_weight
        self.unknown_loss_weight = unknown_loss_weight
        self.physics_warmup_epochs = physics_warmup_epochs
        
        self.current_epoch = tf.Variable(0, trainable=False, dtype=tf.int32)
        
        # Metrics
        self.arcface_loss_tracker = keras.metrics.Mean(name="arcface_loss")
        self.center_loss_tracker = keras.metrics.Mean(name="center_loss")
        self.unknown_loss_tracker = keras.metrics.Mean(name="unknown_loss")
        self.sym_loss_tracker = keras.metrics.Mean(name="sym_loss")
        self.energy_loss_tracker = keras.metrics.Mean(name="energy_loss")
        self.lap_loss_tracker = keras.metrics.Mean(name="lap_loss")
        self.total_loss_tracker = keras.metrics.Mean(name="loss")
        self.accuracy_tracker = keras.metrics.SparseCategoricalAccuracy(name="accuracy")
        self.embed_norm_tracker = keras.metrics.Mean(name="embed_norm")
        
        # Energy-specific metrics
        self.energy_known_tracker = keras.metrics.Mean(name="energy_known")
        self.energy_unknown_tracker = keras.metrics.Mean(name="energy_unknown")
        self.energy_separation_tracker = keras.metrics.Mean(name="energy_sep")
    
    def call(self, inputs):
        return self.base_model(inputs)
    
    def train_step(self, data):
        inputs, labels = data
        face_input = inputs['face_input']
        
        lambda_sym, lambda_energy, lambda_lap = get_physics_weights_tf(
            self.current_epoch,
            warmup_epochs=self.physics_warmup_epochs
        )
        
        with tf.GradientTape() as main_tape:
            # Model outputs: logits, embeddings_norm, embeddings_unnorm
            logits, embeddings, embeddings_unnorm = self.base_model(inputs, training=True)
            
            labels = tf.cast(labels, tf.int32)
            known_mask = tf.greater_equal(labels, 0)
            known_embeddings = tf.boolean_mask(embeddings, known_mask)
            known_embeddings_unnorm = tf.boolean_mask(embeddings_unnorm, known_mask)
            
            # ArcFace loss
            def compute_arcface():
                known_logits = tf.boolean_mask(logits, known_mask)
                known_labels = tf.boolean_mask(labels, known_mask)
                return tf.reduce_mean(
                    tf.nn.sparse_softmax_cross_entropy_with_logits(
                        labels=known_labels,
                        logits=known_logits
                    )
                )
            
            arcface_loss = tf.cond(
                tf.reduce_any(known_mask),
                compute_arcface,
                lambda: tf.constant(0.0, dtype=tf.float32)
            )
            
            # Center loss
            center_loss = self.center_loss_layer(embeddings, labels, training=True)
            
            # Unknown loss
            arcface_layer = self.base_model.get_layer('arcface')
            class_weights = arcface_layer.W
            unknown_loss = self.unknown_rejection_layer(embeddings, labels, class_weights)
            
            # Physics losses
            def compute_physics_losses():
                known_face_input = tf.boolean_mask(face_input, known_mask)
                
                # FIX 3: FULL stop_gradient on augmented inputs
                known_flipped = tf.stop_gradient(tf.image.flip_left_right(known_face_input))
                noise = tf.random.normal(tf.shape(known_face_input), mean=0.0, stddev=0.1)
                known_augmented = tf.stop_gradient(known_face_input + noise)
                
                # Get both normalized and unnormalized embeddings
                embeddings_flip, _ = self.embedding_model(known_flipped, training=True)
                embeddings_aug, _ = self.embedding_model(known_augmented, training=True)
                
                # L_sym (on normalized embeddings)
                diff_sym = known_embeddings - embeddings_flip
                sym = tf.reduce_mean(tf.reduce_sum(tf.square(diff_sym), axis=1))
                
                # L_lap (on normalized embeddings)
                diff_lap = known_embeddings - embeddings_aug
                lap = tf.reduce_mean(tf.reduce_sum(tf.square(diff_lap), axis=1))
                
                return sym, lap
            
            sym_loss, lap_loss = tf.cond(
                tf.reduce_any(known_mask),
                compute_physics_losses,
                lambda: (tf.constant(0.0, dtype=tf.float32),
                        tf.constant(0.0, dtype=tf.float32))
            )
            
            # L_energy: IMPROVED version (requires both known and unknown)
            unknown_mask = tf.less(labels, 0)
            
            def compute_energy_loss():
                """FIX 1: Energy on PRE-L2 embeddings"""
                """FIX 2: Per-sample margin"""
                unknown_emb_unnorm = tf.boolean_mask(embeddings_unnorm, unknown_mask)
                
                energy_loss, E_known, E_unknown = energy_margin_loss_improved(
                    known_embeddings_unnorm, unknown_emb_unnorm, class_weights,
                    margin=CFG.ENERGY_MARGIN, tau=CFG.ENERGY_TAU
                )
                
                # Update energy tracking
                self.energy_known_tracker.update_state(E_known)
                self.energy_unknown_tracker.update_state(E_unknown)
                self.energy_separation_tracker.update_state(E_unknown - E_known)
                
                return energy_loss
            
            has_both = tf.logical_and(
                tf.reduce_any(known_mask),
                tf.reduce_any(unknown_mask)
            )
            
            energy_loss = tf.cond(
                has_both,
                compute_energy_loss,
                lambda: tf.constant(0.0, dtype=tf.float32)
            )
            
            # Total loss
            total_loss = (
                arcface_loss +
                self.center_loss_weight * center_loss +
                self.unknown_loss_weight * unknown_loss +
                lambda_sym * sym_loss +
                lambda_energy * energy_loss +
                lambda_lap * lap_loss
            )
            
            if self.base_model.losses:
                total_loss += tf.add_n(self.base_model.losses)
        
        gradients = main_tape.gradient(total_loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.base_model.trainable_variables))
        
        # Update metrics
        self.arcface_loss_tracker.update_state(arcface_loss)
        self.center_loss_tracker.update_state(center_loss)
        self.unknown_loss_tracker.update_state(unknown_loss)
        self.sym_loss_tracker.update_state(sym_loss)
        self.energy_loss_tracker.update_state(energy_loss)
        self.lap_loss_tracker.update_state(lap_loss)
        self.total_loss_tracker.update_state(total_loss)
        
        embed_norms = tf.norm(embeddings, axis=1)
        self.embed_norm_tracker.update_state(tf.reduce_mean(embed_norms))
        
        def update_acc():
            known_logits = tf.boolean_mask(logits, known_mask)
            known_labels = tf.boolean_mask(labels, known_mask)
            self.accuracy_tracker.update_state(known_labels, known_logits)
            return tf.constant(0)
        
        tf.cond(tf.reduce_any(known_mask), update_acc, lambda: tf.constant(0))
        
        return {
            "loss": self.total_loss_tracker.result(),
            "arcface_loss": self.arcface_loss_tracker.result(),
            "center_loss": self.center_loss_tracker.result(),
            "unknown_loss": self.unknown_loss_tracker.result(),
            "sym_loss": self.sym_loss_tracker.result(),
            "energy_loss": self.energy_loss_tracker.result(),
            "lap_loss": self.lap_loss_tracker.result(),
            "accuracy": self.accuracy_tracker.result(),
            "embed_norm": self.embed_norm_tracker.result(),
            "energy_known": self.energy_known_tracker.result(),
            "energy_unknown": self.energy_unknown_tracker.result(),
            "energy_sep": self.energy_separation_tracker.result()
        }
    
    def test_step(self, data):
        """Validation step"""
        inputs, labels = data
        
        logits, embeddings, _ = self.base_model(inputs, training=False)
        labels = tf.cast(labels, tf.int32)
        
        known_mask = tf.greater_equal(labels, 0)
        
        def compute_val_arcface():
            known_logits = tf.boolean_mask(logits, known_mask)
            known_labels = tf.boolean_mask(labels, known_mask)
            return tf.reduce_mean(
                tf.nn.sparse_softmax_cross_entropy_with_logits(
                    labels=known_labels,
                    logits=known_logits
                )
            )
        
        arcface_loss = tf.cond(
            tf.reduce_any(known_mask),
            compute_val_arcface,
            lambda: tf.constant(0.0, dtype=tf.float32)
        )
        
        center_loss = self.center_loss_layer(embeddings, labels, training=False)
        
        total_loss = arcface_loss + self.center_loss_weight * center_loss
        
        if self.base_model.losses:
            total_loss += tf.add_n(self.base_model.losses)
        
        self.arcface_loss_tracker.update_state(arcface_loss)
        self.center_loss_tracker.update_state(center_loss)
        self.total_loss_tracker.update_state(total_loss)
        
        embed_norms = tf.norm(embeddings, axis=1)
        self.embed_norm_tracker.update_state(tf.reduce_mean(embed_norms))
        
        def update_val_acc():
            known_logits = tf.boolean_mask(logits, known_mask)
            known_labels = tf.boolean_mask(labels, known_mask)
            self.accuracy_tracker.update_state(known_labels, known_logits)
            return tf.constant(0)
        
        tf.cond(tf.reduce_any(known_mask), update_val_acc, lambda: tf.constant(0))
        
        return {
            "loss": self.total_loss_tracker.result(),
            "arcface_loss": self.arcface_loss_tracker.result(),
            "center_loss": self.center_loss_tracker.result(),
            "accuracy": self.accuracy_tracker.result(),
            "embed_norm": self.embed_norm_tracker.result()
        }
    
    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.arcface_loss_tracker,
            self.center_loss_tracker,
            self.unknown_loss_tracker,
            self.sym_loss_tracker,
            self.energy_loss_tracker,
            self.lap_loss_tracker,
            self.accuracy_tracker,
            self.embed_norm_tracker,
            self.energy_known_tracker,
            self.energy_unknown_tracker,
            self.energy_separation_tracker
        ]

trainer = ImprovedPhysicsTrainer(
    base_model=model,
    embedding_model=embedding_model,
    center_loss_layer=center_loss_layer,
    unknown_rejection_layer=unknown_rejection_layer,
    center_loss_weight=CFG.CENTER_LOSS_WEIGHT,
    unknown_loss_weight=CFG.UNKNOWN_LOSS_WEIGHT,
    physics_warmup_epochs=CFG.PHYSICS_WARMUP_EPOCHS
)

print("\n Improved trainer initialized")


 Improved trainer initialized


In [11]:
# CURRICULUM MIXING


def make_curriculum_dataset(known_ds, unknown_ratio):
    """Mix known with synthetic unknowns"""
    if unknown_ratio == 0.0:
        return known_ds.batch(CFG.BATCH_SIZE).prefetch(CFG.AUTOTUNE)
    
    n_known = max(4, int(CFG.BATCH_SIZE * (1.0 - unknown_ratio)))
    n_unknown = CFG.BATCH_SIZE - n_known
    
    print(f"  Batch: {n_known} known + {n_unknown} unknown (heavily augmented)")
    
    known_batched = known_ds.batch(n_known + n_unknown)
    
    def create_mixed_batch(inputs, labels):
        face_imgs = inputs['face_input']
        
        known_faces = face_imgs[:n_known]
        known_labels = labels[:n_known]
        
        # Create unknowns via heavy augmentation
        unknown_source = face_imgs[n_known:n_known + n_unknown]
        unknown_faces = tf.map_fn(augment_unknown, unknown_source, dtype=tf.float32)
        unknown_labels = tf.fill([n_unknown], -1)
        
        merged_faces = tf.concat([known_faces, unknown_faces], axis=0)
        merged_labels = tf.concat([known_labels, unknown_labels], axis=0)
        
        idx = tf.random.shuffle(tf.range(tf.shape(merged_labels)[0]))
        merged_faces = tf.gather(merged_faces, idx)
        merged_labels = tf.gather(merged_labels, idx)
        
        merged_inputs = {
            'face_input': merged_faces,
            'label_input': merged_labels
        }
        
        return merged_inputs, merged_labels
    
    return known_batched.map(create_mixed_batch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [12]:
# CALLBACKS


class EpochCounterCallback(keras.callbacks.Callback):
    def __init__(self, trainer):
        super().__init__()
        self.trainer = trainer
    
    def on_epoch_begin(self, epoch, logs=None):
        self.trainer.current_epoch.assign(epoch)
        
        lambda_sym, lambda_energy, lambda_lap = get_physics_weights_tf(
            epoch,
            warmup_epochs=self.trainer.physics_warmup_epochs
        )
        print(f"\n  [Epoch {epoch+1}] Physics: "
              f"λ_sym={lambda_sym:.5f}, λ_energy={lambda_energy:.5f}, λ_lap={lambda_lap:.5f}")

In [13]:
# TRAINING

print("\n" + "=" * 80)
print("TRAINING")
print("=" * 80)

print(f"\nPhase 1 ({CFG.PHASE_1_EPOCHS} epochs): 100% known")
print(f"Phase 2 ({CFG.PHASE_2_EPOCHS} epochs): 80% known, 20% unknown")
print(f"Phase 3 ({CFG.PHASE_3_EPOCHS} epochs): 60% known, 40% unknown")

# Validation dataset
val_ds = val_data.get_tfdata_unbatched(shuffle=False, augment=False)
val_ds = val_ds.batch(CFG.BATCH_SIZE).prefetch(CFG.AUTOTUNE)
val_steps = int(np.ceil(val_data.total_images / CFG.BATCH_SIZE))

# Build model variables
print("\n✓ Building model variables...")
dummy_batch = next(iter(val_ds))
_ = trainer(dummy_batch[0], training=False)
print("  Model variables created")

trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))

# PHASE 1
print("\n" + "=" * 80)
print(f"PHASE 1: Warmup ({CFG.PHASE_1_EPOCHS} epochs)")
print("=" * 80)

train_ds_p1 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
train_ds_p1 = make_curriculum_dataset(train_ds_p1, unknown_ratio=0.0)

phase1_callbacks = [
    EpochCounterCallback(trainer),
    keras.callbacks.ModelCheckpoint(
        'physics_improved_phase1.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_p1 = trainer.fit(
    train_ds_p1,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=val_steps,
    epochs=CFG.PHASE_1_EPOCHS,
    callbacks=phase1_callbacks
)

# PHASE 2
print("\n" + "=" * 80)
print(f"PHASE 2: Introduction ({CFG.PHASE_2_EPOCHS} epochs)")
print("=" * 80)

train_ds_p2 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
train_ds_p2 = make_curriculum_dataset(train_ds_p2, unknown_ratio=0.2)

trainer.load_weights('physics_improved_phase1.weights.h5')

phase2_callbacks = [
    EpochCounterCallback(trainer),
    keras.callbacks.ModelCheckpoint(
        'physics_improved_phase2.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_p2 = trainer.fit(
    train_ds_p2,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=val_steps,
    initial_epoch=CFG.PHASE_1_EPOCHS,
    epochs=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    callbacks=phase2_callbacks
)

# PHASE 3
print("\n" + "=" * 80)
print(f"PHASE 3: Hardening ({CFG.PHASE_3_EPOCHS} epochs)")
print("=" * 80)

train_ds_p3 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
train_ds_p3 = make_curriculum_dataset(train_ds_p3, unknown_ratio=0.4)

trainer.load_weights('physics_improved_phase2.weights.h5')
trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4))

phase3_callbacks = [
    EpochCounterCallback(trainer),
    keras.callbacks.ModelCheckpoint(
        'physics_improved_final.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_p3 = trainer.fit(
    train_ds_p3,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=val_steps,
    initial_epoch=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    epochs=CFG.TOTAL_EPOCHS,
    callbacks=phase3_callbacks
)

print("TRAINING COMPLETE")


TRAINING

Phase 1 (5 epochs): 100% known
Phase 2 (15 epochs): 80% known, 20% unknown
Phase 3 (10 epochs): 60% known, 40% unknown

✓ Building model variables...


2026-02-22 18:42:27.364375: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8201


  Model variables created

PHASE 1: Warmup (5 epochs)

  [Epoch 1] Physics: λ_sym=0.00000, λ_energy=0.00000, λ_lap=0.00000
Epoch 1/5
200/200 [==============================] - ETA: 0s - loss: 25.2586 - arcface_loss: 25.0822 - center_loss: 0.8084 - unknown_loss: 0.0000e+00 - sym_loss: 0.7578 - energy_loss: 0.0000e+00 - lap_loss: 0.6934 - accuracy: 0.0281 - embed_norm: 1.0000 - energy_known: 0.0000e+00 - energy_unknown: 0.0000e+00 - energy_sep: 0.0000e+00
Epoch 1: val_loss improved from inf to 10.87360, saving model to physics_improved_phase1.weights.h5
200/200 [==============================] - 102s 165ms/step - loss: 25.2586 - arcface_loss: 25.0822 - center_loss: 0.8084 - unknown_loss: 0.0000e+00 - sym_loss: 0.7578 - energy_loss: 0.0000e+00 - lap_loss: 0.6934 - accuracy: 0.0281 - embed_norm: 1.0000 - energy_known: 0.0000e+00 - energy_unknown: 0.0000e+00 - energy_sep: 0.0000e+00 - val_loss: 10.8736 - val_arcface_loss: 10.6878 - val_center_loss: 0.6765 - val_accuracy: 0.2177 - val_embed_

In [14]:
# EXPORT

print("\n Exporting inference model...")

trainer.load_weights('physics_improved_final.weights.h5')

# Inference model outputs only NORMALIZED embeddings
inference_model = keras.Model(
    inputs=trainer.base_model.input[0],
    outputs=trainer.base_model.output[1],  # Normalized embeddings
    name="physics_improved_inference"
)

tf.saved_model.save(inference_model, "physics_improved_saved_model")
print("  Saved: physics_improved_saved_model/")

arcface_layer = model.get_layer('arcface')
class_centers = arcface_layer.W.numpy()
class_centers_norm = class_centers / np.linalg.norm(class_centers, axis=0, keepdims=True)
np.save('physics_improved_class_centers.npy', class_centers_norm.T)
print("  Saved: physics_improved_class_centers.npy")

metadata = {
    'model_type': 'physics_constrained_improved',
    'training_classes': train_data.classes,
    'num_training_classes': train_data.num_classes,
    'data_source': 'face_train + face_val + face_test (combined)',
    'split': f'{CFG.TRAIN_SPLIT*100:.0f}%/{(1-CFG.TRAIN_SPLIT)*100:.0f}% train/val per cow',
    'improvements': [
        'Energy computed on pre-L2 embeddings (stronger separation)',
        'Per-sample energy margin (not batch means)',
        'Full stop_gradient on augmented inputs (stability)'
    ],
    'loss_components': [
        'L_arc: ArcFace angular margin',
        'L_center: Center loss',
        'L_unknown: Unknown rejection',
        'L_sym: Symmetry constraint',
        'L_energy: Improved energy margin (pre-L2, per-sample)',
        'L_lap: Smoothness constraint'
    ],
    'physics_config': {
        'sym_weight': CFG.PHYSICS_SYM_WEIGHT,
        'energy_weight': CFG.PHYSICS_ENERGY_WEIGHT,
        'energy_margin': CFG.ENERGY_MARGIN,
        'lap_weight': CFG.PHYSICS_LAP_WEIGHT,
        'energy_tau': CFG.ENERGY_TAU,
        'warmup_epochs': CFG.PHYSICS_WARMUP_EPOCHS
    }
}

with open('physics_improved_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("  Saved: physics_improved_metadata.json")

print("READY FOR EVALUATION")

print(f"\nModel trained on: {train_data.num_classes} cattle (ALL data combined)")


 Exporting inference model...


INFO:tensorflow:Assets written to: physics_improved_saved_model/assets


INFO:tensorflow:Assets written to: physics_improved_saved_model/assets


  Saved: physics_improved_saved_model/
  Saved: physics_improved_class_centers.npy
  Saved: physics_improved_metadata.json
READY FOR EVALUATION

Model trained on: 468 cattle (ALL data combined)
